In [1]:
# 1. Setup paths and mount Drive
from google.colab import drive
from pathlib import Path
import os

# Mount Drive
drive.mount('/content/drive')

# Path definitions - Drive (persistent)
WORKSPACE = '/content/drive/MyDrive/evo1_finetune'
ASSETS_DIR = Path(f'{WORKSPACE}/Assets')
HDF5_DIR = Path(f'{WORKSPACE}/Trajectories/HDF5')
LEROBOT_DIR = Path(f'{WORKSPACE}/Trajectories/LeRobot')
RESULTS_DIR = Path(f'{WORKSPACE}/Results')

# Path definitions - Local (ephemeral)
CHECKPOINT_DIR = Path('/content/checkpoints/libero')

# Create directories
for d in [ASSETS_DIR, HDF5_DIR, LEROBOT_DIR, RESULTS_DIR, CHECKPOINT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Headless rendering
os.environ['MUJOCO_GL'] = 'egl'
os.environ.pop('DISPLAY', None)

print(f'✅ MUJOCO_GL: {os.environ.get("MUJOCO_GL")}')

print(f'✅ Workspace: {WORKSPACE}')
print(f'   Results: {RESULTS_DIR}')

print(f'   Assets: {ASSETS_DIR}')
print(f'   LeRobot: {LEROBOT_DIR}')
print(f'   HDF5: {HDF5_DIR}')

Mounted at /content/drive
✅ MUJOCO_GL: egl
✅ Workspace: /content/drive/MyDrive/evo1_finetune
   Results: /content/drive/MyDrive/evo1_finetune/Results
   Assets: /content/drive/MyDrive/evo1_finetune/Assets
   LeRobot: /content/drive/MyDrive/evo1_finetune/Trajectories/LeRobot
   HDF5: /content/drive/MyDrive/evo1_finetune/Trajectories/HDF5


In [ ]:
# 2. Install all dependencies
!apt-get update -qq && apt-get install -y -qq git wget ffmpeg libosmesa6-dev libgl1-mesa-glx libegl1-mesa > /dev/null 2>&1

# Install Miniconda
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
!bash /tmp/miniconda.sh -b -p /opt/conda > /dev/null 2>&1 && rm /tmp/miniconda.sh
os.environ['PATH'] = f"/opt/conda/bin:{os.environ['PATH']}"
!conda init bash --quiet && conda config --set always_yes yes --quiet

# VLABench environment (Python 3.10) - Official requirements
print('📦 Creating VLABench environment...')
!conda create -n vlabench python=3.10 -y -q
!conda run -n vlabench pip install -q \
  lerobot==0.3.2 \
  'datasets>=2.19.0,<=3.6.0' \
  draccus==0.10.0 \
  'rerun-sdk>=0.21.0,<0.23.0' \
  'numpy>=1.24,<2.0' \
  mujoco dm_control gymnasium==0.29.1 \
  scipy tqdm colorlog colorama open3d \
  h5py av zarr pillow opencv-python-headless mediapy

# Evo-1 environment (Python 3.10) - Official requirements
print('📦 Creating Evo-1 environment...')
!conda create -n evo1 python=3.10 -y -q
!conda run -n evo1 pip install -q \
  torch==2.5.1+cu121 torchvision==0.20.1+cu121 \
  --index-url https://download.pytorch.org/whl/cu121

# Clone and install Evo-1
print('📦 Cloning Evo-1...')
!git clone -q https://github.com/MINT-SJTU/Evo-1.git /content/Evo-1 2>/dev/null || true

# Install Evo-1 requirements (OFFICIAL method)
print('📦 Installing Evo-1 requirements...')
!conda run -n evo1 pip install -q -r /content/Evo-1/Evo_1/requirements.txt

# Install Flash-Attention (CRITICAL - required for Evo-1)
print('📦 Installing Flash-Attention (CRITICAL - may take 10-15 min)...')
!conda run -n evo1 bash -c "MAX_JOBS=4 pip install flash-attn --no-build-isolation" 2>&1 | tail -3

# Clone VLABench and dependencies
print('📦 Cloning VLABench...')
!git clone -q https://github.com/OpenMOSS/VLABench.git /content/VLABench 2>/dev/null || true
!git clone -q https://github.com/motion-planning/rrt-algorithms.git /content/rrt-algorithms 2>/dev/null || true

# Install VLABench (OFFICIAL method)
print('📦 Installing VLABench...')
!conda run -n vlabench pip install -q -e /content/rrt-algorithms
!conda run -n vlabench pip install -q -e /content/VLABench

# Setup assets
print('📦 Setting up assets...')
if not (ASSETS_DIR / 'obj').exists():
    !conda run -n vlabench python /content/VLABench/scripts/download_assets.py
    !mv /content/VLABench/VLABench/assets/obj /content/VLABench/VLABench/assets/scenes {ASSETS_DIR}/

!rm -rf /content/VLABench/VLABench/assets/obj /content/VLABench/VLABench/assets/scenes
!ln -s {ASSETS_DIR}/obj /content/VLABench/VLABench/assets/obj
!ln -s {ASSETS_DIR}/scenes /content/VLABench/VLABench/assets/scenes

# Download checkpoint
print('📦 Downloading checkpoint...')
from huggingface_hub import snapshot_download
if not (CHECKPOINT_DIR / 'mp_rank_00_model_states.pt').exists():
    snapshot_download(repo_id='MINT-SJTU/Evo1_LIBERO', local_dir=str(CHECKPOINT_DIR), local_dir_use_symlinks=False)

print('✅ Setup complete!')
print('⚠️  Flash-Attention is CRITICAL for Evo-1 - ensure it installed successfully')


📦 Installing system dependencies...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
📦 Installing Miniconda...
no change     /opt/conda/condabin/conda
no change     /opt/conda/bin/conda
no change     /opt/conda/bin/conda-env
no change     /opt/conda/bin/activate
no change     /opt/conda/bin/deactivate
no change     /opt/conda/etc/profile.d/conda.sh
no change     /opt/conda/etc/fish/conf.d/conda.fish
no change     /opt/conda/shell/condabin/Conda.psm1
no change     /opt/conda/shell/condabin/conda-hook.ps1
no change     /opt/conda/lib/python3.13/site-packages/xontrib/conda.xsh
no change     /opt/conda/etc/profile.d/conda.csh
modified      /root/.bashrc

==> For changes to take effect, close and re-open your current shell. <==

accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:979: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

mp_rank_00_model_states.pt:   0%|          | 0.00/1.55G [00:00<?, ?B/s]

norm_stats.json:   0%|          | 0.00/894 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/990 [00:00<?, ?B/s]

checkpoint.json:   0%|          | 0.00/89.0 [00:00<?, ?B/s]

In [ ]:
# 3. Generate trajectories
import subprocess
import os

TASKS = ['select_toy', 'select_fruit', 'select_drink']

print(f'🎯 Generating trajectories for {len(TASKS)} tasks')

for task in TASKS:
    task_dir = HDF5_DIR / task
    task_dir.mkdir(exist_ok=True, parents=True)
    
    # Check existing
    existing = len(list(task_dir.glob('*.hdf5')))
    if existing >= 20:
        print(f'✅ {task}: {existing} trajectories (skipped)')
        continue
    
    print(f'🔄 {task}...')
    
    # Generate
    subprocess.run([
        'conda', 'run', '-n', 'vlabench',
        'python', '/content/VLABench/scripts/trajectory_generation.py',
        '--task-name', task,
        '--n-sample', '50',
        '--max-episode', '20',
        '--save-dir', str(task_dir),
        '--robot', 'franka',
        '--record-video', 'true'
    ], env={**os.environ, 'MUJOCO_GL': 'egl'})
    
    count = len(list(task_dir.glob('*.hdf5')))
    print(f'✅ {task}: {count} trajectories')

total = sum(len(list((HDF5_DIR / t).glob('*.hdf5'))) for t in TASKS)
print(f'\n✅ Total: {total} trajectories')


🎯 Generating trajectories for 3 tasks
   Local workspace: /content/temp_trajectories
   Final destination: /content/drive/MyDrive/vlabench/hdf5

🔄 Generating select_toy...


In [ ]:
# 4. Convert to LeRobot format
print('🔄 Converting to LeRobot format...')

subprocess.run([
    'conda', 'run', '-n', 'vlabench',
    'python', '/content/VLABench/scripts/convert_to_lerobot.py',
    '--dataset-name', 'vlabench_finetune',
    '--dataset-path', str(HDF5_DIR),
    '--task-list', *TASKS,
    '--max-files', '500'
])

print('✅ Conversion complete!')


In [ ]:
# 5. Fine-tune Evo-1 (TODO: Implement)
print('⚠️ Fine-tuning not yet implemented')
print(f'📂 Dataset: {LEROBOT_DIR}/vlabench_finetune')
print(f'📂 Checkpoint: {CHECKPOINT_DIR}')
print(f'📂 Output: {RESULTS_DIR}')


🔧 Fine-tuning Evo-1...
⚠️  This cell is a placeholder - implement your fine-tuning script here


NameError: name 'RESULTS_DIR' is not defined